# Business Case #1 — Data-Driven Client Segmentation for Financial Services

**Politecnico di Milano — Machine Learning for FinTech Lab**

---

## Objective

This notebook addresses the problem of **segmenting bank clients** into meaningful groups (**Financial Personas**) using unsupervised machine learning techniques. The goal is to move beyond traditional segmentation criteria (e.g., age or wealth alone) and discover data-driven customer prototypes that capture multidimensional behavioral and demographic patterns.

## Dataset

We work with a dataset of **5,000 anonymized bank clients**, each described by **17 features** spanning demographics (age, gender, job, geographic area), financial attributes (income, wealth, debt), and behavioral propensities (digital, ESG, saving, lifestyle, etc.). The data contains a **mix of numerical and categorical variables**, which is the central technical challenge.

## Methodological Pipeline

1. **Data Loading & Exploratory Data Analysis (EDA)**
2. **Preprocessing** — handling mixed data types (one-hot encoding, scaling)
3. **Distance Metrics** — Gower distance for mixed-type data, custom mixed distance
4. **Dimensionality Reduction** — t-SNE (with multiple metrics) and PCA for visualization
5. **Clustering** — K-Medoids (with Gower distance), K-Means, Hierarchical, DBSCAN
6. **Cluster Evaluation** — Silhouette, Calinski-Harabasz, Davies-Bouldin, voting scheme
7. **Cluster Interpretation & Persona Definition** — profiling, heatmaps, radar charts
8. **New Client Assignment** — assigning incoming clients to existing personas
9. **Bayesian Persona Update** — refining personas as new individual data arrives

## Technical Note on Mixed Data

Standard clustering algorithms like K-Means rely on Euclidean distance and mean-based centroids, which are ill-defined for categorical features. We adopt two complementary strategies:
- **Gower distance** combined with **K-Medoids** (medoids are actual data points, bypassing the mean problem)
- **One-hot encoding + scaling** combined with standard algorithms and a **custom mixed distance** (Hamming for categorical + Manhattan for numerical, following the Matlab reference)

This notebook synthesizes the strengths of both approaches presented in class, consolidating them into a single, coherent, and reproducible workflow.

---

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist, squareform, hamming, cityblock
from mpl_toolkits.mplot3d import Axes3D

import warnings
warnings.filterwarnings('ignore')

# For reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
print("All libraries loaded successfully.")

---
## 2. Data Loading & Initial Inspection

We load the dataset and the metadata file, then perform a first inspection to understand the structure, types, and distributions of the features.

In [ ]:
# Load the main dataset
df = pd.read_excel('Dataset1_BankClients.xlsx')

# Load metadata for reference
metadata = pd.read_excel('BankClients_Metadata.xlsx')

print("=== Dataset Shape ===")
print(f"{df.shape[0]} clients, {df.shape[1]} columns\n")

print("=== Column Types ===")
print(df.dtypes.to_string())

print("\n=== First 5 Records ===")
display(df.head())

print("\n=== Metadata (Feature Descriptions) ===")
display(metadata)

In [ ]:
# Drop the ID column — it is not informative for clustering
data = df.drop(columns=['ID']).copy()

print("=== Descriptive Statistics ===")
display(data.describe().round(3))

print("\n=== Missing Values ===")
print(data.isnull().sum().to_string())

---
## 3. Exploratory Data Analysis (EDA)

Before diving into clustering, we visualize the distributions and relationships in the data. This helps us build intuition about potential cluster structures and identify any data quality issues.

### 3.1 Feature Distributions

In [ ]:
# Mapping labels for categorical features (for interpretability)
job_labels = {1: 'Unemployed', 2: 'Employee', 3: 'Manager', 4: 'Entrepreneur', 5: 'Retired'}
area_labels = {1: 'Nord', 2: 'Centro', 3: 'Sud/Isole'}
inv_labels = {1: 'No investments', 2: 'Lump sum', 3: 'Capital accumulation'}
city_labels = {1: 'Small town', 2: 'Medium city', 3: 'Large city'}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Dataset Overview', fontsize=16, fontweight='bold')

# Age distribution by gender
sns.histplot(data=data, x='Age', hue='Gender', multiple='stack', bins=25, ax=axes[0, 0],
             palette={0: 'steelblue', 1: 'coral'})
axes[0, 0].set_title('Age Distribution by Gender')
axes[0, 0].legend(labels=['Male (0)', 'Female (1)'])

# Job distribution
data['Job'].map(job_labels).value_counts().plot(kind='bar', ax=axes[0, 1], color='steelblue', edgecolor='black')
axes[0, 1].set_title('Job Distribution')
axes[0, 1].tick_params(axis='x', rotation=25)

# Income vs Wealth scatter
axes[0, 2].scatter(data['Income'], data['Wealth'], alpha=0.08, s=8, color='darkorange')
axes[0, 2].set_xlabel('Income (percentile)')
axes[0, 2].set_ylabel('Wealth (percentile)')
axes[0, 2].set_title('Income vs Wealth')

# Investment type
data['Investments'].map(inv_labels).value_counts().plot(
    kind='pie', ax=axes[1, 0], autopct='%1.1f%%', colors=['#ff9999', '#66b3ff', '#99ff99'])
axes[1, 0].set_title('Investment Type')
axes[1, 0].set_ylabel('')

# Digital vs Financial Education
axes[1, 1].scatter(data['Digital'], data['FinEdu'], alpha=0.08, s=8, color='green')
axes[1, 1].set_xlabel('Digital Propensity')
axes[1, 1].set_ylabel('Financial Education')
axes[1, 1].set_title('Digital vs Financial Education')

# Geographical area
data['Area'].map(area_labels).value_counts().plot(kind='bar', ax=axes[1, 2], color='purple', edgecolor='black')
axes[1, 2].set_title('Geographical Area')
axes[1, 2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

### 3.2 The Categorical Variable Problem

A fundamental challenge in this dataset is the **coexistence of categorical and numerical features**. Standard distance metrics (Euclidean, Manhattan) treat all features as continuous, which is meaningless for categories like Gender or Job. The scatter plot below illustrates why: categorical variables create discrete bands that can dominate clustering and obscure the underlying continuous structure.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gender vs Age — illustrates discrete banding
axes[0].scatter(data['Age'], data['Gender'], alpha=0.3, s=10, color='steelblue')
axes[0].set_xlabel('Age (numerical, continuous)')
axes[0].set_ylabel('Gender (categorical, nominal)')
axes[0].set_title('Why categorical variables are problematic for clustering')
axes[0].set_yticks([0, 1])
axes[0].set_yticklabels(['Male', 'Female'])

# Correlation heatmap of numerical features
numerical_cols = ['Age', 'Income', 'Wealth', 'Debt', 'FinEdu', 'ESG',
                  'Digital', 'BankFriend', 'LifeStyle', 'Luxury', 'Saving', 'FamilySize']
corr = data[numerical_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, ax=axes[1], cbar_kws={'shrink': 0.7})
axes[1].set_title('Correlation Matrix (Numerical Features)')

plt.tight_layout()
plt.show()

---
## 4. Preprocessing

We prepare two representations of the data, corresponding to the two distance strategies:

**Representation A** — For Gower distance + K-Medoids:
- We keep the original data structure, simply marking which columns are categorical. The Gower distance handles mixed types natively.

**Representation B** — For standard algorithms (K-Means, Hierarchical, etc.):
- **Unordered categorical** features (Job, Area, Investments) → One-Hot Encoding (drop first to avoid multicollinearity)
- **Ordered categorical** features (Age, CitySize, FamilySize) → MinMaxScaler to [0, 1]
- **Already normalized** features (Income, Wealth, etc.) → taken as-is (already in [0, 1] percentiles)
- **Gender** → binary, taken as-is

In [ ]:
# ============================================================
# REPRESENTATION A: For Gower distance (original data, typed)
# ============================================================
# The gower library requires categorical columns to be of 'object' (string) dtype.
# NOTE: some versions of gower also accept 'category', but 'object' is more robust.
categorical_columns = ['Gender', 'Job', 'Area', 'CitySize', 'Investments']
data_for_gower = data.copy()
for col in categorical_columns:
    data_for_gower[col] = data_for_gower[col].astype(str)

print("Representation A (for Gower): original data with string-typed categoricals")
print(f"  Shape: {data_for_gower.shape}")
print(f"  Categorical columns: {categorical_columns}")
print(f"  Dtypes: {data_for_gower[categorical_columns].dtypes.to_dict()}")

# ============================================================
# REPRESENTATION B: For standard algorithms (encoded + scaled)
# ============================================================
# Unordered categorical → One-Hot Encoding
unordered_cat_cols = ['Job', 'Area', 'Investments']
encoder = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
X_unordered = encoder.fit_transform(data[unordered_cat_cols])
unordered_names = encoder.get_feature_names_out(unordered_cat_cols).tolist()

# Ordered categorical → MinMaxScaler
ordered_cat_cols = ['Age', 'CitySize', 'FamilySize']
scaler_ordered = MinMaxScaler()
X_ordered = scaler_ordered.fit_transform(data[ordered_cat_cols])

# Already normalized features (percentiles in [0,1]) + Gender (binary)
already_scaled_cols = ['Gender', 'Income', 'Wealth', 'Debt', 'FinEdu',
                       'ESG', 'Digital', 'BankFriend', 'LifeStyle', 'Luxury', 'Saving']
X_scaled = data[already_scaled_cols].values

# Final feature matrix
X = np.hstack([X_scaled, X_ordered, X_unordered])
all_feature_names = already_scaled_cols + ordered_cat_cols + unordered_names
df_encoded = pd.DataFrame(X, columns=all_feature_names)

print(f"\nRepresentation B (encoded + scaled): {X.shape[0]} samples, {X.shape[1]} features")
print(f"  Already scaled: {len(already_scaled_cols)} features")
print(f"  Ordered scaled:  {len(ordered_cat_cols)} features")
print(f"  One-hot encoded: {len(unordered_names)} features")
display(df_encoded.head())

---
## 5. Distance Computation

### 5.1 Gower Distance

The **Gower distance** is specifically designed for mixed data. It combines:
- **Absolute difference** (scaled to [0,1]) for numerical variables
- **Jaccard distance** (0 if equal, 1 if different) for categorical variables
- The final distance is the **weighted average** across all features

This makes it the natural metric for our heterogeneous dataset.

In [ ]:
# Install gower if needed (uncomment the line below if running in Colab/Jupyter)
# !pip install gower

import gower

# Compute the full Gower distance matrix on original data
# NOTE: this is an O(n^2) operation — may take ~30s for 5000 samples
print("Computing Gower distance matrix (5000 x 5000)... this may take a moment.")

# The gower library expects categorical columns as 'object' dtype (strings).
# Numerical columns should remain as float/int.
# data_for_gower was prepared in the preprocessing step with this convention.
cat_features_mask = [col in categorical_columns for col in data_for_gower.columns]
gower_dist_matrix = gower.gower_matrix(data_for_gower, cat_features=cat_features_mask)

print(f"Gower distance matrix shape: {gower_dist_matrix.shape}")
print(f"Min distance: {gower_dist_matrix[gower_dist_matrix > 0].min():.4f}")
print(f"Max distance: {gower_dist_matrix.max():.4f}")
print(f"Sample distances (first 5x5):\n{np.round(gower_dist_matrix[:5, :5], 4)}")

### 5.2 Custom Mixed Distance (Hamming + Manhattan)

Following the approach from the Matlab reference (`MixDistance.m`), we implement a custom distance that:
- Uses **Hamming distance** for one-hot encoded categorical features (measures the proportion of mismatched binary entries)
- Uses **Manhattan (L1) distance** for numerical features
- Combines them with a **weighted average**, where the weight reflects the proportion of categorical vs. numerical dimensions

This provides an alternative to Gower and gives us control over the distance decomposition.

In [ ]:
def mixed_distance_matrix(X, n_cat_features):
    """
    Compute a mixed distance matrix combining:
    - Hamming distance for the first n_cat_features columns (one-hot encoded categoricals)
    - Manhattan (L1) distance for the remaining columns (numericals)
    
    The two distances are combined as a weighted average, where the weight 
    reflects the proportion of categorical vs numerical dimensions.
    
    Parameters
    ----------
    X : np.ndarray of shape (n_samples, n_features)
    n_cat_features : int
        Number of leading columns that are categorical (one-hot encoded)
    
    Returns
    -------
    D : np.ndarray of shape (n_samples, n_samples)
    """
    from scipy.spatial.distance import pdist, squareform
    
    X_cat = X[:, :n_cat_features]
    X_num = X[:, n_cat_features:]
    
    D_cat = squareform(pdist(X_cat, metric='hamming'))
    D_num = squareform(pdist(X_num, metric='cityblock'))
    
    # Normalize numerical distance to [0, 1] range for comparability
    if D_num.max() > 0:
        D_num = D_num / D_num.max()
    
    n_total = X.shape[1]
    w_cat = n_cat_features / n_total
    w_num = 1.0 - w_cat
    
    D = w_cat * D_cat + w_num * D_num
    return D

# For the encoded representation, identify how many columns are one-hot (categorical)
# The one-hot columns are at the END of X: they come from Job (4 dummies), Area (2), Investments (2) = 8 total
n_ohe_features = X_unordered.shape[1]
# But we also need to decide: are Gender, CitySize treated as categorical for distance?
# Following the Matlab approach: binary/dummy columns first, numerical after.
# Let's reorder: [one-hot features] + [Gender] + [ordered + numerical]

# Reorder X so that all binary/categorical features come first
X_cat_block = np.hstack([X_unordered, X_scaled[:, 0:1]])  # OHE + Gender
X_num_block = np.hstack([X_scaled[:, 1:], X_ordered])       # Numerical + ordered
X_mixed = np.hstack([X_cat_block, X_num_block])
n_cat_total = X_cat_block.shape[1]  # one-hot + Gender

print(f"Mixed distance layout: {n_cat_total} categorical cols + {X_num_block.shape[1]} numerical cols")
print("Computing custom mixed distance matrix...")
D_mixed = mixed_distance_matrix(X_mixed, n_cat_total)
print(f"Mixed distance matrix shape: {D_mixed.shape}")
print(f"Sample distances (first 5x5):\n{np.round(D_mixed[:5, :5], 4)}")

---
## 6. Dimensionality Reduction & Visualization

Since we cannot directly visualize a 20+ dimensional space, we use dimensionality reduction to project the data into 2D and 3D. This serves as a diagnostic tool to detect potential cluster structures.

### 6.1 t-SNE with Multiple Distance Metrics

We compare four different distance metrics in t-SNE to see how the choice of metric affects the visible cluster structure. This mirrors the approach from the Matlab code (`Project1.m`), which compared city-block, cosine, Euclidean, and a custom mixed distance.

In [ ]:
from sklearn.utils import check_random_state

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('t-SNE 2D Embeddings with Different Distance Metrics', fontsize=16, fontweight='bold')

# Standard metrics on encoded data
metrics_standard = [
    ('cityblock', 'Manhattan (City Block)'),
    ('cosine', 'Cosine'),
    ('euclidean', 'Euclidean'),
]

for idx, (metric, title) in enumerate(metrics_standard):
    row, col = divmod(idx, 2)
    tsne = TSNE(n_components=2, metric=metric, random_state=RANDOM_STATE,
                init='random', learning_rate='auto', perplexity=30)
    Y = tsne.fit_transform(X)
    axes[row, col].scatter(Y[:, 0], Y[:, 1], alpha=0.3, s=5, c='steelblue')
    axes[row, col].set_title(title, fontsize=13)
    axes[row, col].grid(True, alpha=0.3)
    print(f"  t-SNE with {title} completed.")

# Gower distance (precomputed)
tsne_gower = TSNE(n_components=2, metric='precomputed', random_state=RANDOM_STATE,
                  init='random', learning_rate='auto', perplexity=30)
Y_gower_2d = tsne_gower.fit_transform(gower_dist_matrix)
axes[1, 1].scatter(Y_gower_2d[:, 0], Y_gower_2d[:, 1], alpha=0.3, s=5, c='darkorange')
axes[1, 1].set_title('Gower Distance', fontsize=13)
axes[1, 1].grid(True, alpha=0.3)
print("  t-SNE with Gower completed.")

plt.tight_layout()
plt.show()

### 6.2 3D t-SNE Embedding (Gower Distance)

A 3D embedding can reveal structure that is collapsed in 2D projections.

In [ ]:
# 3D t-SNE with Gower distance
tsne_3d = TSNE(n_components=3, metric='precomputed', random_state=RANDOM_STATE,
               init='random', learning_rate='auto', perplexity=30)
Y_gower_3d = tsne_3d.fit_transform(gower_dist_matrix)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(Y_gower_3d[:, 0], Y_gower_3d[:, 1], Y_gower_3d[:, 2],
           alpha=0.4, s=5, c='teal', edgecolors='k', linewidths=0.1)
ax.set_title('3D t-SNE Embedding (Gower Distance)', fontsize=14, fontweight='bold')
ax.view_init(elev=20, azim=-40)
plt.tight_layout()
plt.show()

### 6.3 PCA — Linear Dimensionality Reduction

PCA provides a complementary (linear) perspective. It also tells us how much variance is captured by each component, which is informative about the intrinsic dimensionality of the data.

In [ ]:
# PCA on the encoded representation
pca_full = PCA()
X_pca_full = pca_full.fit_transform(X)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Explained variance
axes[0].bar(range(1, len(pca_full.explained_variance_ratio_) + 1),
            pca_full.explained_variance_ratio_, alpha=0.7, color='steelblue', label='Individual')
axes[0].step(range(1, len(pca_full.explained_variance_ratio_) + 1),
             np.cumsum(pca_full.explained_variance_ratio_), where='mid', color='red', label='Cumulative')
axes[0].set_title('Explained Variance by PCA Components')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2D PCA
axes[1].scatter(X_pca_full[:, 0], X_pca_full[:, 1], alpha=0.15, s=5, c='crimson')
axes[1].set_title('2D PCA Embedding')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].grid(True, alpha=0.3)

# Cumulative variance — how many components for 90%?
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_90 = np.argmax(cumvar >= 0.90) + 1
axes[2].plot(range(1, len(cumvar) + 1), cumvar, 'o-', color='steelblue')
axes[2].axhline(y=0.90, color='red', linestyle='--', label=f'90% variance → {n_90} components')
axes[2].set_title('Cumulative Explained Variance')
axes[2].set_xlabel('Number of Components')
axes[2].set_ylabel('Cumulative Variance')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Components needed for 90% variance: {n_90}")
print(f"Total explained variance (first 3 PCs): {cumvar[2]:.2%}")

---
## 7. Clustering

We apply multiple clustering algorithms to compare results and build confidence in our segmentation.

### 7.1 K-Medoids with Gower Distance

K-Medoids is the natural partner for Gower distance: it uses **actual data points** as cluster centers (medoids), which is well-defined for mixed data types — unlike K-Means, which computes means that are undefined for categorical features.

In [ ]:
# Install scikit-learn-extra for KMedoids (uncomment if needed)
# !pip install scikit-learn-extra

from sklearn_extra.cluster import KMedoids

# Run K-Medoids for k = 3 to 7 with Gower distance
kmedoids_results = {}
k_range_med = range(3, 8)

print("Running K-Medoids with Gower distance...")
for k in k_range_med:
    kmed = KMedoids(n_clusters=k, metric='precomputed', random_state=RANDOM_STATE,
                    init='k-medoids++', max_iter=300)
    labels = kmed.fit_predict(gower_dist_matrix)
    kmedoids_results[k] = labels
    sil = silhouette_score(gower_dist_matrix, labels, metric='precomputed')
    print(f"  k={k}: Silhouette={sil:.4f}")

### 7.2 K-Means on Encoded Data

We also apply K-Means on the one-hot encoded + scaled representation. While K-Means is not ideal for mixed data, it is a common baseline and can still produce useful results on properly preprocessed data.

In [ ]:
# Run K-Means for k = 3 to 10
kmeans_results = {}
inertia_list = []
sil_kmeans_list = []
k_range_km = range(2, 11)

print("Running K-Means on encoded data...")
for k in k_range_km:
    km = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    labels = km.fit_predict(X)
    kmeans_results[k] = labels
    inertia_list.append(km.inertia_)
    sil = silhouette_score(X, labels)
    sil_kmeans_list.append(sil)
    print(f"  k={k}: Inertia={km.inertia_:.1f}, Silhouette={sil:.4f}")

### 7.3 Cluster Evaluation

We use three complementary metrics to determine the optimal number of clusters:

- **Silhouette Score** — measures how similar each point is to its own cluster vs. the nearest other cluster (higher is better)
- **Calinski-Harabasz Index** — ratio of between-cluster to within-cluster dispersion (higher is better)
- **Davies-Bouldin Index** — average similarity between clusters (lower is better)

We evaluate both K-Means and K-Medoids results.

In [ ]:
# --- K-Means evaluation ---
ch_km = [calinski_harabasz_score(X, kmeans_results[k]) for k in k_range_km]
db_km = [davies_bouldin_score(X, kmeans_results[k]) for k in k_range_km]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Cluster Evaluation Metrics', fontsize=16, fontweight='bold')

# Elbow plot
axes[0, 0].plot(list(k_range_km), inertia_list, 'o-', color='steelblue', linewidth=2)
axes[0, 0].set_title('K-Means: Elbow Method (Inertia)')
axes[0, 0].set_xlabel('k')
axes[0, 0].set_ylabel('Inertia')
axes[0, 0].grid(True, alpha=0.3)

# Silhouette — K-Means
best_sil_km = list(k_range_km)[np.argmax(sil_kmeans_list)]
axes[0, 1].plot(list(k_range_km), sil_kmeans_list, 'o-', color='darkorange', linewidth=2)
axes[0, 1].axvline(x=best_sil_km, color='red', linestyle='--', alpha=0.7, label=f'Best k={best_sil_km}')
axes[0, 1].set_title('K-Means: Silhouette Score')
axes[0, 1].set_xlabel('k')
axes[0, 1].set_ylabel('Silhouette')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Calinski-Harabasz — K-Means
axes[1, 0].plot(list(k_range_km), ch_km, 'o-', color='green', linewidth=2)
axes[1, 0].set_title('K-Means: Calinski-Harabasz Index')
axes[1, 0].set_xlabel('k')
axes[1, 0].set_ylabel('CH Index')
axes[1, 0].grid(True, alpha=0.3)

# Davies-Bouldin — K-Means
axes[1, 1].plot(list(k_range_km), db_km, 'o-', color='crimson', linewidth=2)
axes[1, 1].set_title('K-Means: Davies-Bouldin Index')
axes[1, 1].set_xlabel('k')
axes[1, 1].set_ylabel('DB Index (lower is better)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- K-Medoids evaluation ---
sil_med = [silhouette_score(gower_dist_matrix, kmedoids_results[k], metric='precomputed') for k in k_range_med]
ch_med = [calinski_harabasz_score(X, kmedoids_results[k]) for k in k_range_med]
db_med = [davies_bouldin_score(X, kmedoids_results[k]) for k in k_range_med]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
fig.suptitle('K-Medoids (Gower) Evaluation', fontsize=14, fontweight='bold')

axes[0].plot(list(k_range_med), sil_med, 'go-', linewidth=2)
axes[0].set_title('Silhouette (Gower)')
axes[0].set_xlabel('k')
axes[0].grid(True, alpha=0.3)

axes[1].plot(list(k_range_med), ch_med, 'bo-', linewidth=2)
axes[1].set_title('Calinski-Harabasz')
axes[1].set_xlabel('k')
axes[1].grid(True, alpha=0.3)

axes[2].plot(list(k_range_med), db_med, 'ro-', linewidth=2)
axes[2].set_title('Davies-Bouldin (lower is better)')
axes[2].set_xlabel('k')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 7.4 Optimal k Selection via Voting

Since different metrics may suggest different optimal values of k, we implement a **voting scheme**: each metric casts a vote for its preferred k, and we take the **median** as the consensus decision. This approach reduces the influence of any single metric's bias.

In [ ]:
def voting_optimal_k(X, cluster_results, k_range, gower_dist=None):
    """Determine optimal k by voting across three metrics."""
    ch_scores = [calinski_harabasz_score(X, cluster_results[k]) for k in k_range]
    db_scores = [davies_bouldin_score(X, cluster_results[k]) for k in k_range]
    
    if gower_dist is not None:
        sil_scores = [silhouette_score(gower_dist, cluster_results[k], metric='precomputed') for k in k_range]
    else:
        sil_scores = [silhouette_score(X, cluster_results[k]) for k in k_range]
    
    k_list = list(k_range)
    k_ch = k_list[np.argmax(ch_scores)]
    k_db = k_list[np.argmin(db_scores)]
    k_sil = k_list[np.argmax(sil_scores)]
    
    optimal_k = int(np.median([k_ch, k_db, k_sil]))
    
    print("=== Voting Results ===")
    print(f"  Calinski-Harabasz suggests: k = {k_ch}")
    print(f"  Davies-Bouldin suggests:    k = {k_db}")
    print(f"  Silhouette suggests:        k = {k_sil}")
    print(f"  --> Consensus (median):     k = {optimal_k}")
    
    return optimal_k

print("--- K-Medoids (Gower) ---")
optimal_k_med = voting_optimal_k(X, kmedoids_results, k_range_med, gower_dist=gower_dist_matrix)

print("\n--- K-Means (Encoded) ---")
# Use only k=3..7 range for fair comparison
kmeans_sub = {k: kmeans_results[k] for k in range(3, 8)}
optimal_k_km = voting_optimal_k(X, kmeans_sub, range(3, 8))

### 7.5 Hierarchical Clustering

Agglomerative (bottom-up) hierarchical clustering provides an alternative perspective. The dendrogram helps visualize the merging process and can suggest a natural number of clusters.

In [ ]:
# Dendrogram on a subsample (full 5000 is too dense to visualize)
np.random.seed(RANDOM_STATE)
sample_idx = np.random.choice(len(X), 300, replace=False)
X_sample = X[sample_idx]

linked = linkage(X_sample, method='ward')

plt.figure(figsize=(14, 5))
dendrogram(linked, truncate_mode='lastp', p=30, leaf_rotation=90,
           leaf_font_size=8, show_contracted=True)
plt.title('Dendrogram — Ward Linkage (subsample of 300)', fontsize=13, fontweight='bold')
plt.xlabel('Client (or merged cluster)')
plt.ylabel('Distance')
plt.tight_layout()
plt.show()

# Compare with K-Means
K_FINAL = optimal_k_med  # Use the voting result
agglo = AgglomerativeClustering(n_clusters=K_FINAL, linkage='ward')
labels_agglo = agglo.fit_predict(X)

sil_agglo = silhouette_score(X, labels_agglo)
sil_km_final = silhouette_score(X, kmeans_results[K_FINAL]) if K_FINAL in kmeans_results else 0
sil_med_final = silhouette_score(gower_dist_matrix, kmedoids_results[K_FINAL], metric='precomputed') if K_FINAL in kmedoids_results else 0

print(f"\n=== Comparison at k = {K_FINAL} ===")
print(f"  K-Medoids (Gower) Silhouette: {sil_med_final:.4f}")
print(f"  K-Means (encoded)  Silhouette: {sil_km_final:.4f}")
print(f"  Hierarchical (Ward) Silhouette: {sil_agglo:.4f}")

### 7.6 DBSCAN — Density-Based Clustering

DBSCAN discovers clusters of arbitrary shape and does not require specifying k in advance. Instead, it relies on two parameters: `eps` (neighborhood radius) and `min_samples` (minimum density threshold). Points that don't belong to any dense region are labeled as **noise** (-1).

We use the precomputed Gower distance matrix.

In [ ]:
# DBSCAN with Gower distance — scan eps values
eps_values = np.linspace(0.04, 0.16, 7)

print("DBSCAN exploration (Gower distance):")
print(f"{'eps':>8} | {'Clusters':>8} | {'Noise pts':>9} | {'Silhouette':>10}")
print("-" * 45)

dbscan_results = {}
for eps in eps_values:
    db = DBSCAN(eps=eps, min_samples=10, metric='precomputed')
    labels_db = db.fit_predict(gower_dist_matrix)
    n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
    n_noise = (labels_db == -1).sum()
    
    sil = None
    if n_clusters >= 2 and n_noise < len(labels_db) * 0.5:
        # Only compute silhouette if meaningful
        non_noise = labels_db != -1
        if len(set(labels_db[non_noise])) >= 2:
            sil = silhouette_score(gower_dist_matrix[non_noise][:, non_noise],
                                   labels_db[non_noise], metric='precomputed')
    
    dbscan_results[eps] = {'labels': labels_db, 'n_clusters': n_clusters,
                           'n_noise': n_noise, 'silhouette': sil}
    sil_str = f"{sil:.4f}" if sil is not None else "N/A"
    print(f"{eps:>8.3f} | {n_clusters:>8} | {n_noise:>9} | {sil_str:>10}")

# Visualization of DBSCAN results
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(eps_values, [dbscan_results[e]['n_clusters'] for e in eps_values], 'bo-')
axes[0].set_title('Number of Clusters vs. eps')
axes[0].set_xlabel('eps')
axes[0].set_ylabel('Clusters found')
axes[0].grid(True, alpha=0.3)

axes[1].plot(eps_values, [dbscan_results[e]['n_noise'] for e in eps_values], 'ro-')
axes[1].set_title('Noise Points vs. eps')
axes[1].set_xlabel('eps')
axes[1].set_ylabel('Noise points')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 8. Final Clustering & Persona Interpretation

Based on the evaluation metrics and visual inspection, we select the final number of clusters and apply K-Medoids with Gower distance as our primary segmentation method. We then perform a comprehensive analysis to interpret each cluster as a **Financial Persona**.

> **Adjust `K_FINAL` below if your analysis suggests a different value.**

In [ ]:
# ============================================================
# FINAL CLUSTERING — K-Medoids with Gower distance
# ============================================================
# Adjust K_FINAL based on your evaluation
K_FINAL = optimal_k_med
print(f"Final number of clusters: K = {K_FINAL}")

kmed_final = KMedoids(n_clusters=K_FINAL, metric='precomputed',
                      random_state=RANDOM_STATE, init='k-medoids++', max_iter=300)
final_labels = kmed_final.fit_predict(gower_dist_matrix)

# Add labels to original data (remove old Cluster column if re-running)
if 'Cluster' in data.columns:
    data = data.drop(columns=['Cluster'])
data['Cluster'] = final_labels

# Cluster sizes
print("\n=== Cluster Sizes ===")
for c in sorted(data['Cluster'].unique()):
    n = (data['Cluster'] == c).sum()
    print(f"  Cluster {c}: {n} clients ({n/len(data)*100:.1f}%)")

### 8.1 Cluster Profiles — Statistical Summary

We compute the mean (for numerical features) and mode/distribution (for categorical features) of each cluster to build interpretable profiles.

In [ ]:
# --- Numerical feature profiles ---
numerical_profile_cols = ['Age', 'Income', 'Wealth', 'Debt', 'FinEdu', 'ESG',
                          'Digital', 'BankFriend', 'LifeStyle', 'Luxury', 'Saving', 'FamilySize']
cluster_profile = data.groupby('Cluster')[numerical_profile_cols].mean().round(3)

print("=== Cluster Centroids (numerical features) ===")
display(cluster_profile)

# --- Categorical feature distributions ---
cat_display_maps = {
    'Gender': {0: 'Male', 1: 'Female'},
    'Job': job_labels,
    'Area': area_labels,
    'CitySize': city_labels,
    'Investments': inv_labels,
}

for var, label_map in cat_display_maps.items():
    ct = pd.crosstab(data['Cluster'], data[var].map(label_map), normalize='index') * 100
    print(f"\n=== {var} Distribution per Cluster (%) ===")
    display(ct.round(1))

### 8.2 Feature Heatmap

A normalized heatmap allows us to compare clusters across all numerical features at a glance. Values are scaled to [0, 1] across clusters (row-min / row-max), with the actual averages shown as annotations.

In [ ]:
heatmap_data = cluster_profile.copy()
heatmap_norm = (heatmap_data - heatmap_data.min()) / (heatmap_data.max() - heatmap_data.min() + 1e-9)

plt.figure(figsize=(15, 0.8 * K_FINAL + 2))
sns.heatmap(heatmap_norm, annot=heatmap_data.values.round(2), fmt='.2f',
            cmap='RdYlGn', linewidths=0.5, cbar_kws={'label': 'Normalized value'},
            xticklabels=heatmap_data.columns, yticklabels=[f'Cluster {i}' for i in range(K_FINAL)])
plt.title('Cluster Profiles — Feature Heatmap', fontsize=14, fontweight='bold')
plt.xlabel('Features')
plt.ylabel('')
plt.tight_layout()
plt.show()

### 8.3 Radar Chart

Radar (spider) charts provide an intuitive overlay of cluster profiles, making it easy to see the distinctive "shape" of each persona.

In [ ]:
radar_features = ['Income', 'Wealth', 'FinEdu', 'Digital', 'ESG',
                  'Luxury', 'Saving', 'BankFriend', 'Debt', 'LifeStyle']

radar_data = cluster_profile[radar_features]
radar_norm = (radar_data - radar_data.min()) / (radar_data.max() - radar_data.min() + 1e-9)

N_feat = len(radar_features)
angles = np.linspace(0, 2 * np.pi, N_feat, endpoint=False).tolist()
angles += angles[:1]

colors = plt.cm.tab10.colors

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
fig.suptitle('Cluster Radar Chart', fontsize=14, fontweight='bold', y=0.98)

for i in range(K_FINAL):
    values = radar_norm.iloc[i].tolist() + [radar_norm.iloc[i].tolist()[0]]
    ax.plot(angles, values, linewidth=2, color=colors[i], label=f'Cluster {i}')
    ax.fill(angles, values, alpha=0.08, color=colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_features, fontsize=10)
ax.set_ylim(0, 1)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1))
ax.grid(True)
plt.tight_layout()
plt.show()

### 8.4 Cluster Visualizations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
fig.suptitle('Cluster Analysis — Key Dimensions', fontsize=16, fontweight='bold')
cluster_sizes = data['Cluster'].value_counts().sort_index()

# Cluster sizes (pie)
axes[0, 0].pie(cluster_sizes.values, labels=[f'Cl. {i}' for i in cluster_sizes.index],
               autopct='%1.1f%%', colors=colors[:K_FINAL])
axes[0, 0].set_title('Cluster Sizes')

# Age by cluster
for i in range(K_FINAL):
    axes[0, 1].hist(data[data['Cluster'] == i]['Age'], bins=20, alpha=0.5,
                    label=f'Cl. {i}', color=colors[i])
axes[0, 1].set_title('Age Distribution')
axes[0, 1].set_xlabel('Age')
axes[0, 1].legend(fontsize=8)

# Income vs Wealth
for i in range(K_FINAL):
    mask = data['Cluster'] == i
    axes[0, 2].scatter(data[mask]['Income'], data[mask]['Wealth'],
                       alpha=0.15, s=8, color=colors[i], label=f'Cl. {i}')
axes[0, 2].set_title('Income vs Wealth')
axes[0, 2].set_xlabel('Income')
axes[0, 2].set_ylabel('Wealth')
axes[0, 2].legend(fontsize=8)

# Digital vs FinEdu
for i in range(K_FINAL):
    mask = data['Cluster'] == i
    axes[1, 0].scatter(data[mask]['Digital'], data[mask]['FinEdu'],
                       alpha=0.15, s=8, color=colors[i], label=f'Cl. {i}')
axes[1, 0].set_title('Digital vs Financial Education')
axes[1, 0].set_xlabel('Digital')
axes[1, 0].set_ylabel('FinEdu')
axes[1, 0].legend(fontsize=8)

# Gender ratio by cluster
gender_ratio = data.groupby('Cluster')['Gender'].mean()
gender_ratio.plot(kind='bar', ax=axes[1, 1], color=colors[:K_FINAL], edgecolor='black')
axes[1, 1].set_title('Female Ratio by Cluster')
axes[1, 1].axhline(y=0.5, color='red', linestyle='--', alpha=0.6)
axes[1, 1].set_ylabel('Proportion Female')
axes[1, 1].tick_params(axis='x', rotation=0)

# Investment type by cluster
inv_ct = pd.crosstab(data['Cluster'], data['Investments'].map(inv_labels), normalize='index')
inv_ct.plot(kind='bar', stacked=True, ax=axes[1, 2])
axes[1, 2].set_title('Investment Type by Cluster')
axes[1, 2].set_ylabel('Proportion')
axes[1, 2].tick_params(axis='x', rotation=0)
axes[1, 2].legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.show()

### 8.5 t-SNE Visualization Colored by Cluster

In [ ]:
# 2D and 3D t-SNE colored by final cluster labels
fig = plt.figure(figsize=(16, 6))

# 2D plot
ax_2d = fig.add_subplot(121)
for i in range(K_FINAL):
    mask = final_labels == i
    ax_2d.scatter(Y_gower_2d[mask, 0], Y_gower_2d[mask, 1],
                    alpha=0.3, s=8, color=colors[i], label=f'Cluster {i}')
ax_2d.set_title('2D t-SNE (Gower) — Colored by Cluster')
ax_2d.legend(fontsize=8)
ax_2d.grid(True, alpha=0.3)

# 3D plot
ax_3d = fig.add_subplot(122, projection='3d')
for i in range(K_FINAL):
    mask = final_labels == i
    ax_3d.scatter(Y_gower_3d[mask, 0], Y_gower_3d[mask, 1], Y_gower_3d[mask, 2],
                 alpha=0.3, s=8, color=colors[i], label=f'Cluster {i}')
ax_3d.set_title('3D t-SNE (Gower)')
ax_3d.legend(fontsize=8)
ax_3d.view_init(elev=20, azim=-40)

plt.tight_layout()
plt.show()

### 8.6 Financial Personas — Summary Table

Based on the cluster profiles above, fill in the persona descriptions. This is the **qualitative overlay** step: use the data to name each cluster and infer its main financial needs and preferred service model.

| Cluster | Persona Name | Key Demographic Traits | Financial Profile | Main Needs | Service Model |
|---------|-------------|----------------------|-------------------|------------|---------------|
| 0 | *(e.g., Young Digital)* | ... | ... | ... | Digital |
| 1 | *(e.g., Wealthy Retiree)* | ... | ... | ... | Physical |
| ... | ... | ... | ... | ... | ... |

> **Instructions**: Examine the heatmap, radar chart, and categorical distributions above. Identify the distinguishing features of each cluster and assign meaningful persona names. Consider what financial products and services would best serve each persona.

---
## 9. Assigning New Clients to Personas

Once the personas are established, any new client can be assigned to the closest persona using the same preprocessing pipeline and distance computation. Below is a demonstration with a synthetic new client.

In [ ]:
# Example: new client data (using original feature values)
new_client = pd.DataFrame([{
    'Age': 45, 'Gender': 1, 'Job': 3, 'Area': 1, 'CitySize': 3,
    'FamilySize': 2, 'Income': 0.75, 'Wealth': 0.80, 'Debt': 0.20,
    'FinEdu': 0.85, 'ESG': 0.70, 'Digital': 0.90, 'BankFriend': 0.65,
    'LifeStyle': 0.75, 'Luxury': 0.60, 'Saving': 0.55, 'Investments': 3
}])

# Prepare new client in the same format as data_for_gower
new_for_gower = new_client.copy()
for col in categorical_columns:
    new_for_gower[col] = new_for_gower[col].astype(str)

# data_for_gower does NOT contain 'Cluster' — it was created before clustering.
# Combine with original gower-formatted data to compute distances.
combined = pd.concat([data_for_gower, new_for_gower], ignore_index=True)
cat_mask = [col in categorical_columns for col in combined.columns]
gower_new = gower.gower_matrix(combined, cat_features=cat_mask)

# Distance from the new client (last row) to all existing clients
dist_new = gower_new[-1, :-1]

# Assign to the cluster whose medoid is closest
medoid_indices = kmed_final.medoid_indices_
distances_to_medoids = dist_new[medoid_indices]
assigned_cluster = np.argmin(distances_to_medoids)

print(f"New client assigned to: Cluster {assigned_cluster}")
print(f"\nDistances to medoids: {dict(zip(range(K_FINAL), np.round(distances_to_medoids, 4)))}")
print(f"\nCluster {assigned_cluster} profile:")
display(cluster_profile.loc[assigned_cluster].to_frame().T)

---
## 10. Bayesian Update of Financial Personas

In practice, a client is first assigned to a persona (the **prior**), and then their profile is gradually refined as individual data arrives. This section demonstrates two Bayesian updating strategies:

### 10.1 Univariate Beta-Binomial Update

Each numerical feature of the persona is modeled as a **Beta distribution** on [0, 1]. The persona's feature value becomes the initial mean, and as new observations arrive, the distribution is updated via the Beta-Binomial conjugate model. The distribution narrows over time, reflecting growing confidence in the individual's true value.

In [ ]:
import scipy.stats as stats

# Select a persona vector (numerical features only, already in [0,1])
# Using the profile of the assigned cluster
persona_vector = cluster_profile.loc[assigned_cluster][
    ['Income', 'Wealth', 'Debt', 'FinEdu', 'ESG', 'Digital', 'BankFriend', 'LifeStyle', 'Luxury', 'Saving']
].values.astype(float)

feature_names_bayes = ['Income', 'Wealth', 'Debt', 'FinEdu', 'ESG',
                       'Digital', 'BankFriend', 'LifeStyle', 'Luxury', 'Saving']
K_feat = len(persona_vector)

# Clamp persona values to avoid Beta distribution issues at 0 or 1
persona_vector = np.clip(persona_vector, 0.01, 0.99)

# Initial "sample size" — represents how much we trust the persona prior
N_prior = 20
alpha_param = persona_vector * N_prior
beta_param = (1 - persona_vector) * N_prior

# Simulate 3 sequential data updates (in practice, these come from real observations)
np.random.seed(42)
updates = np.random.uniform(0.1, 0.9, (3, K_feat))

def bayesian_beta_update(a, b, new_obs, N):
    """Update Beta parameters with a new observation."""
    return a + new_obs * N, b + (1 - new_obs) * N

def plot_beta_distributions(a_params, b_params, names, title):
    fig, axes = plt.subplots(2, 5, figsize=(20, 6))
    x = np.linspace(0.001, 0.999, 200)
    for idx, ax in enumerate(axes.flat):
        if idx < len(a_params):
            y = stats.beta.pdf(x, a_params[idx], b_params[idx])
            ax.plot(x, y, color='steelblue', linewidth=2)
            ax.fill_between(x, y, color='steelblue', alpha=0.2)
            mean_val = a_params[idx] / (a_params[idx] + b_params[idx])
            ax.axvline(mean_val, color='red', linestyle='--', alpha=0.7, label=f'\u03bc={mean_val:.3f}')
            ax.set_title(names[idx], fontsize=10)
            ax.set_xlim(0, 1)
            ax.legend(fontsize=7)
        else:
            ax.set_visible(False)
    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Plot initial distributions (persona prior)
plot_beta_distributions(alpha_param, beta_param, feature_names_bayes,
                        f"Prior: Persona {assigned_cluster} (N={N_prior})")

# Sequential updates
for t, update in enumerate(updates):
    alpha_param, beta_param = bayesian_beta_update(alpha_param, beta_param, update, N_prior)
    plot_beta_distributions(alpha_param, beta_param, feature_names_bayes,
                            f"Posterior after update {t+1}")

print("As new data arrives, the distributions narrow — reflecting reduced uncertainty about the individual's profile.")

### 10.2 Multivariate Gaussian Update (Kalman Filter)

The univariate approach treats each feature independently, ignoring correlations. A more sophisticated alternative uses a **Multivariate Gaussian model** with a **Kalman filter** update. This accounts for inter-feature dependencies (e.g., high income correlating with high wealth).

The key idea:
1. Transform persona values from [0,1] to ℝ via the **logit** function
2. Model the prior as MVN(μ₀, Σ₀)
3. Update using the Kalman gain: μ' = μ₀ + K(y - μ₀)
4. Transform back to [0,1] via the **sigmoid** function

In [ ]:
def logit(x, eps=1e-6):
    """Logit transform, clipped to avoid infinity."""
    x = np.clip(x, eps, 1 - eps)
    return np.log(x / (1 - x))

def sigmoid(z):
    """Inverse logit (sigmoid)."""
    return 1 / (1 + np.exp(-z))

# Re-load persona_vector (it may have been modified by the Beta update above)
persona_vector_mvn = cluster_profile.loc[assigned_cluster][
    ['Income', 'Wealth', 'Debt', 'FinEdu', 'ESG', 'Digital', 'BankFriend', 'LifeStyle', 'Luxury', 'Saving']
].values.astype(float)
persona_vector_mvn = np.clip(persona_vector_mvn, 0.01, 0.99)

# Initial persona in logit space
mu_0 = logit(persona_vector_mvn)

# Covariance matrix — in practice, estimate from the data
# Here we use a simple structure: uniform std + moderate correlation
std_dev = np.full(K_feat, 0.4)
rho = 0.3  # off-diagonal correlation
corr_mat = np.full((K_feat, K_feat), rho)
np.fill_diagonal(corr_mat, 1.0)
Sigma_0 = np.outer(std_dev, std_dev) * corr_mat
Sigma_0 += np.eye(K_feat) * 1e-6  # numerical stability

# Noise covariance for observations
Sigma_obs = np.eye(K_feat) * 0.15

def kalman_update(mu, Sigma, y_new, Sigma_obs):
    """Standard Kalman filter update."""
    K_gain = Sigma @ np.linalg.inv(Sigma + Sigma_obs)
    mu_new = mu + K_gain @ (y_new - mu)
    Sigma_new = (np.eye(len(mu)) - K_gain) @ Sigma
    return mu_new, Sigma_new

def plot_persona_bar(values, names, title):
    plt.figure(figsize=(10, 4))
    bars = plt.bar(range(len(values)), values, color='steelblue', alpha=0.8, edgecolor='black')
    plt.xticks(range(len(values)), names, rotation=30, ha='right')
    plt.ylim(0, 1)
    plt.title(title, fontsize=13, fontweight='bold')
    plt.ylabel('Value (0-1 scale)')
    for bar, val in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.2f}', ha='center', fontsize=8)
    plt.tight_layout()
    plt.show()

# Plot initial persona
plot_persona_bar(persona_vector_mvn, feature_names_bayes,
                 f"Initial Persona {assigned_cluster}")

# Sequential Kalman updates
mu = mu_0.copy()
Sigma = Sigma_0.copy()
np.random.seed(42)
updates_mvn = np.random.uniform(0.1, 0.9, (3, K_feat))

for t, update in enumerate(updates_mvn):
    y = logit(update)
    mu, Sigma = kalman_update(mu, Sigma, y, Sigma_obs)
    updated_persona = sigmoid(mu)
    plot_persona_bar(updated_persona, feature_names_bayes,
                     f"Persona after Multivariate Update {t+1}")

print("The multivariate approach accounts for correlations between features,")
print("producing more coherent updates when features are interdependent.")

---
## 11. Summary & Conclusions

This notebook presented a comprehensive pipeline for **data-driven client segmentation** in financial services, addressing the key challenge of **mixed-type data** (categorical + numerical).

### Key Methodological Contributions

1. **Distance metrics matter**: We compared Gower distance, Euclidean, Manhattan, and a custom Hamming+Manhattan mix. Gower is the most principled choice for heterogeneous data.

2. **Multiple clustering algorithms**: K-Medoids (ideal for Gower), K-Means (common baseline), Hierarchical (dendrogram insight), and DBSCAN (density-based, automatic k detection). Each has different strengths.

3. **Robust k selection**: A voting scheme across Silhouette, Calinski-Harabasz, and Davies-Bouldin metrics provides a consensus that is more robust than relying on any single metric.

4. **Interpretability**: Heatmaps, radar charts, and statistical profiles transform abstract clusters into actionable **Financial Personas** with clear business semantics.

5. **Operational deployment**: New client assignment and Bayesian persona updating demonstrate how the segmentation can be used in a live business context — starting from a persona prior and progressively refining as individual data accumulates.

### Hints for Further Exploration

- **Try other dimensionality reduction techniques**: UMAP, PACMAP, ICA, Factor Analysis
- **Tune t-SNE perplexity**: use a logarithmic grid search to find the best value
- **Experiment with feature weighting**: give higher weight to features you consider more business-relevant
- **Try Self-Organizing Maps (SOM)**: a neural network approach to clustering (see `SOM_Clustering.ipynb`)
- **Explore feature subsets**: remove features that add noise, re-cluster, and compare results
- **Apply Occam's Razor**: a simpler model that explains the data well is preferable to an unnecessarily complex one

---
*Notebook prepared for the Machine Learning for FinTech Lab — Politecnico di Milano*